# ML Model Benchmark Suite — Tutorial

This notebook walks through the core features of the benchmark suite:

1. Loading datasets (CSV and sklearn built-ins)
2. Running a benchmark experiment
3. Inspecting results and metrics
4. Generating HTML reports
5. Querying experiment history
6. Comparing two experiment runs

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on the path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from benchmark.runner import BenchmarkRunner
from benchmark.report import ReportGenerator
from benchmark.tracking import ExperimentTracker
from benchmark.compare import ExperimentComparator
from benchmark.data import load_dataset
from benchmark.registry import REGISTRY

## 1. List available models

In [ ]:
models = REGISTRY.list_models()
for name, meta in sorted(models.items()):
    print(f"  - {name} ({meta['type']})")

## 2. Load a dataset

In [ ]:
dataset_config = {
    "source": "csv",
    "path": str(project_root / "datasets" / "iris.csv"),
    "target_column": "target",
}

X, y = load_dataset(dataset_config)
print(f"Loaded: X={X.shape}, y={y.shape}")
X.head()

## 3. Run a benchmark experiment from a config file

In [ ]:
config_path = project_root / "config" / "example.yaml"
runner = BenchmarkRunner(str(config_path))
results = runner.run()

print(f"Experiment: {results['experiment_name']}")
print(f"Run ID: {results['run_id']}")
print(f"Models: {results['models']}")
print(f"Status: {results['status']}")

## 4. Inspect per-model metrics

In [ ]:
for model_name, model_res in results["results"].items():
    print(f"\n=== {model_name} ===")
    agg = model_res["aggregated"]
    for metric, value in agg.get("val", {}).items():
        print(f"  {metric}: {value:.4f}")
    overfitting = model_res.get("overfitting", {})
    print(f"  Overfitting status: {overfitting.get('status', 'unknown')}")
    if overfitting.get("warnings"):
        for w in overfitting["warnings"]:
            print(f"    ⚠ {w}")

## 5. Generate an HTML report

In [ ]:
report_path = project_root / "reports" / f"tutorial_report_{results['run_id']}.html"
gen = ReportGenerator()

sklearn_models = {
    name: wrapper.model
    for name, wrapper in runner.model_instances.items()
}

gen.generate(
    results,
    str(report_path),
    X=runner.X_processed,
    y=runner.y_processed,
    models=sklearn_models,
)
print(f"Report saved to: {report_path}")

## 6. Query experiment history

In [ ]:
tracker = ExperimentTracker()
runs = tracker.list_runs(limit=10)

print(f"{'ID':<5} {'Experiment':<30} {'Status':<12} {'Timestamp'}")
print("-" * 70)
for run in runs:
    print(f"{run['id']:<5} {run['experiment_name']:<30} {run['status']:<12} {run['timestamp']}")

## 7. Compare two recent runs

In [ ]:
if len(runs) >= 2:
    run_id_a = runs[0]["id"]
    run_id_b = runs[1]["id"]
    comparator = ExperimentComparator()
    comparison = comparator.compare(run_id_a, run_id_b)
    if comparison:
        compare_path = project_root / "reports" / f"compare_{run_id_a}_vs_{run_id_b}.html"
        comparator.generate_report(comparison, str(compare_path))
        print(f"Comparison report saved to: {compare_path}")
    else:
        print("Could not generate comparison.")
else:
    print("Need at least two runs to compare.")

## 8. Next steps

- Try the `config/bank.yaml` or `config/tuning_test.yaml` configs.
- Explore the CLI with `python main.py --history`.
- Export results with `python main.py --export-json` and `--export-csv`.